In [1]:
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

/Users/johnson/Documents/article-dev/fine-tuning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "outputs/final-adapter"
PROMPTS = "../data/benchmark_prompts.jsonl"
OUT = "../data/eval_results.jsonl"

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, ADAPTER_DIR).to(device)
model.eval()
model.config.use_cache = True

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 18095.05it/s]


In [5]:

EMOJI = re.compile(
    "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F02F]"
)

In [14]:
# def generate(messages):
#     inputs = tokenizer.apply_chat_template(
#         messages, add_generation_prompt=True, return_tensors="pt"
#     ).to(device)
#     with torch.no_grad():
#         out = model.generate(
#             inputs, max_new_tokens=120, do_sample=False,  # greedy = reproducible
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

def generate(messages):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        out[0][input_length:],
        skip_special_tokens=True,
    ).strip()

def auto_metrics(text):
    """Cheap proxies for the ShopEase style (NOT a substitute for judging)."""
    words = len(text.split())
    return {
        "words": words,
        "has_emoji": bool(EMOJI.search(text)),
        "is_brief": words <= 60,   # dataset replies are short
    }

In [15]:
prompts = [json.loads(l) for l in open(PROMPTS)]
rows = []
for p in prompts:
    msgs = p["messages"]

    with model.disable_adapter():       # <-- baseline: pure base model
        base_reply = generate(msgs)
    ft_reply = generate(msgs)            # <-- fine-tuned: adapter active

    rows.append({
        "id": p["id"], "category": p["category"],
        "user": msgs[-1]["content"],
        "baseline": base_reply, "baseline_metrics": auto_metrics(base_reply),
        "fine_tuned": ft_reply, "fine_tuned_metrics": auto_metrics(ft_reply),
    })
    print(f"[{p['id']:>2}] {p['category']}: done")

with open(OUT, "w") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")


def agg(key):
        emoji = sum(r[key]["has_emoji"] for r in rows)
        brief = sum(r[key]["is_brief"] for r in rows)
        avglen = sum(r[key]["words"] for r in rows) / len(rows)
        return emoji, brief, avglen
    
for label, key in [("BASELINE", "baseline_metrics"), ("FINE-TUNED", "fine_tuned_metrics")]:
    e, b, l = agg(key)
    print(f"{label:>10}: emoji {e}/{len(rows)}  brief {b}/{len(rows)}  avg_words {l:.1f}")
print(f"\nWrote {OUT}. Now score quality against benchmark_references.jsonl.")

[ 1] shipping: done
[ 2] shipping: done
[ 3] shipping: done
[ 4] returns: done
[ 5] returns: done
[ 6] returns: done
[ 7] billing: done
[ 8] billing: done
[ 9] billing: done
[10] account: done
[11] account: done
[12] product: done
[13] product: done
[14] product: done
[15] order_status: done
[16] order_status: done
[17] complaint: done
[18] complaint: done
[19] edge_case: done
[20] edge_case: done
  BASELINE: emoji 0/20  brief 16/20  avg_words 48.1
FINE-TUNED: emoji 14/20  brief 19/20  avg_words 25.4

Wrote ../data/eval_results.jsonl. Now score quality against benchmark_references.jsonl.
